<div align="center">

# Universidad Autónoma de Chihuahua
## Facultad de Ingeniería
### Maestría en Ingeniería en Computación

**Materia:** Machine Learning (MIC1)

**Actividad:** Proyecto Final — Notebook 5: Ejecución completa y verificación

**Alumno:** Jonahtan Gabriel González Cardoza
**Matrícula:** 345491

**Catedrática:** Dra. Olanda Prieto Ordaz

**Fecha de elaboración:** 21 de noviembre de 2026

</div>

**Proyecto:** predicción de la situación final de un estudiante (*Dropout*, *Enrolled*, *Graduate*) con el dataset
*Predict Students' Dropout and Academic Success* (UCI 697). Este proyecto reúne, en cinco notebooks reproducibles, el trabajo
de las tres prácticas anteriores (regresión logística del Proyecto F_1, SVM y métodos de ensamble).

**Objetivo de este notebook.** Ejecutar los notebooks 01 → 04 en orden, con el mismo kernel de Python, guardar las copias ejecutadas en `outputs/ejecutados/` y comprobar que las métricas coinciden con las que se reportaron en las prácticas anteriores (`referencia.json`).

## Cómo usarlo
1. Revisa los parámetros de la celda siguiente.
2. **Run All** (o `Kernel → Restart & Run All`). Tarda unos minutos (la mayor parte es la optimización de hiperparámetros del notebook 03 y la validación cruzada repetida del 04).
3. Al final aparecen la tabla de tiempos y la tabla de verificación contra `referencia.json`.

Si un notebook falla, el error se muestra con la celda que lo causó; corrige y vuelve a ejecutar este notebook (o ejecuta ese notebook solo y continúa con los siguientes).

In [ ]:
import json, os, sys, time
from pathlib import Path

import nbformat
import pandas as pd
from nbclient import NotebookClient

RAIZ = Path.cwd()
assert (RAIZ / "config.py").exists(), "Abre este notebook desde la carpeta del proyecto (donde está config.py)"

NOTEBOOKS = ["01_configuracion_inicial.ipynb", "02_eda.ipynb", "03_modelos.ipynb", "04_resultados.ipynb"]
SALTAR = []                    # p. ej. ["01_configuracion_inicial.ipynb"] si ya los ejecutaste y solo quieres repetir los siguientes
MODO_RAPIDO = False            # True solo para depurar (reduce búsquedas y remuestreos; los resultados NO son los del proyecto)
TOLERANCIA = 0.005             # diferencia máxima aceptada contra referencia.json (EBM y AdaBoost varían en la 3.ª–4.ª cifra entre máquinas)

if MODO_RAPIDO:
    os.environ["PROYECTO_RAPIDO"] = "1"
else:
    os.environ.pop("PROYECTO_RAPIDO", None)

EJECUTADOS = RAIZ / "outputs" / "ejecutados"
EJECUTADOS.mkdir(parents=True, exist_ok=True)
print("Carpeta del proyecto:", RAIZ)

In [ ]:
tiempos = {}
for nombre in NOTEBOOKS:
    if nombre in SALTAR:
        print(f"— {nombre}: omitido")
        continue
    nb = nbformat.read(RAIZ / nombre, as_version=4)
    kernel = nb.metadata.get("kernelspec", {}).get("name", "python3")
    cliente = NotebookClient(nb, timeout=None, kernel_name=kernel, resources={"metadata": {"path": str(RAIZ)}})
    t0 = time.perf_counter()
    print(f"▶ {nombre} ...", flush=True)
    try:
        cliente.execute()
    finally:                                   # se guarda la copia aunque falle, para ver dónde se detuvo
        nbformat.write(nb, EJECUTADOS / nombre)
    tiempos[nombre] = time.perf_counter() - t0
    print(f"  ✓ {nombre}: {tiempos[nombre] / 60:.1f} min", flush=True)

print("\nTiempos de ejecución (min):")
display(pd.Series({k: round(v / 60, 2) for k, v in tiempos.items()}, name="min").to_frame())

## Verificación contra los resultados ya reportados
Se compara lo que produjo esta corrida con `referencia.json` (F1 macro en CV y en Test de cada modelo, y el tamaño de la partición).
**OK** = diferencia ≤ tolerancia. **REVISAR** = diferencia mayor: suele deberse a otra versión de scikit-learn / interpret o a haber cambiado algo en `config.py`.

In [ ]:
import joblib
sys.path.insert(0, str(RAIZ))
from config import MODELOS, TABLAS, cargar_particion, NOMBRE_LR

ref = json.load(open(RAIZ / "referencia.json", encoding="utf-8"))
filas = []

def comparar(concepto, obtenido, esperado, tol=TOLERANCIA):
    dif = obtenido - esperado
    filas.append({"Concepto": concepto, "Esperado": esperado, "Obtenido": round(float(obtenido), 4),
                  "Diferencia": round(float(dif), 4), "Estado": "OK" if abs(dif) <= tol else "REVISAR"})

# Partición
X_train, X_test, y_train, y_test = cargar_particion()
comparar("Estudiantes en Train", len(X_train), ref["particion"]["train"], tol=0)
comparar("Estudiantes en Test", len(X_test), ref["particion"]["test"], tol=0)

# F1 macro en CV (5 folds)
registro = joblib.load(MODELOS / "registro.joblib")
for modelo, esperado in ref["cv_f1"].items():
    comparar(f"F1 macro CV — {modelo}", registro["resultados_cv"][modelo]["F1_mean"], esperado)
comparar("Threshold de la LR", registro["extra"]["umbral_lr"], ref["umbral_lr"], tol=0)

# F1 macro y Accuracy en Test
test = pd.read_csv(TABLAS / "04_test_todos_los_modelos.csv", index_col=0)
for modelo, esperado in ref["test_f1"].items():
    comparar(f"F1 macro Test — {modelo}", test.loc[modelo, "F1"], esperado)
for modelo, esperado in ref["test_accuracy"].items():
    comparar(f"Accuracy Test — {modelo}", test.loc[modelo, "Accuracy"], esperado)

verif = pd.DataFrame(filas)
display(verif.style.hide(axis="index").map(lambda v: "background-color: #d7f0dc" if v == "OK" else ("background-color: #fbd5d5" if v == "REVISAR" else ""), subset=["Estado"]))
n_rev = int((verif["Estado"] == "REVISAR").sum())
print(f"\n{len(verif) - n_rev} de {len(verif)} comprobaciones OK" + ("" if n_rev == 0 else f" — {n_rev} para revisar"))
verif.to_csv(TABLAS / "05_verificacion.csv", index=False, encoding="utf-8-sig")